In [63]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Sankey dataframe

In [41]:
from sklearn.cluster import KMeans
import numpy as np

In [42]:
import numpy as np
from sklearn.cluster import KMeans

def detect_label_switching(
        data : np.ndarray,
        num_clusters_range: range
        ) -> np.ndarray:
    """
    Detects label switching in KMeans clustering from one iteration to the next.

    Parameters
    ----------
    data : numpy.ndarray
        One-dimensional array of data points to cluster.
    num_clusters_range : range
        Range of number of clusters to try.
    
    Returns
    -------
    numpy.ndarray
        Array of labels for each data point for each iteration of KMeans clustering.
    """
    prev_labels = None  # Initialize previous labels as None
    km_labels = np.zeros((data.shape[0],data.shape[0]), dtype=int)
    km_labels_memorize = np.zeros((data.shape[0],data.shape[0]), dtype=int)

    for num_clusters in num_clusters_range:
        kmeans = KMeans(n_clusters=data.shape[0] - num_clusters, random_state=0, n_init=10)
        kmeans.fit(data.reshape(-1, 1))
        labels = kmeans.labels_
        km_labels[num_clusters] = labels
        if num_clusters == 0:
            km_labels_memorize[num_clusters] = labels
        print('previous iteration', prev_labels)
        if prev_labels is not None:
            print('num_cluster is', num_clusters)
            # Compare current labels with previous labels
            switched_indices = np.where(labels != prev_labels)[0]
            print('switched indices', switched_indices)
            if len(switched_indices) > 0:
                #print(f"Label switching detected at iteration {num_clusters - 1} to {num_clusters}:")
                for index in switched_indices:
                    prev_label = prev_labels[index]
                    curr_label = labels[index]
                    #print(f"Data point at index {index}: switched from label {prev_label} to label {curr_label}")
            km_labels_memorize[num_clusters] = km_labels_memorize[num_clusters - 1]
            print('before changing previous iteration', km_labels_memorize[num_clusters])
            for i, index in enumerate(switched_indices):
                print('subiteration is', labels[index])
                print('TEST', np.where(km_labels_memorize[num_clusters-1] == km_labels_memorize[num_clusters-1][index]))
                print(np.where(km_labels_memorize[num_clusters-1] == km_labels_memorize[num_clusters-1][index])[0])
                num_repeated_labels = len(np.where(km_labels_memorize[num_clusters-1] == km_labels_memorize[num_clusters-1][index])[0])
                print('give number of repeated label', num_repeated_labels)
                if  num_repeated_labels>1 and prev_labels is not None and num_clusters>0:
                    print("here")
                    all_index = np.where(km_labels[num_clusters] == labels[index])[0]
                    km_labels_memorize[num_clusters][all_index] = int(np.max(km_labels_memorize[num_clusters - 1])) + i + 1
        
            
        prev_labels = labels
    return km_labels, km_labels_memorize

# Example usage:
data = np.array([0.59830198, 0.7101328 , 0.25210179, 0.90204906, 0.79986231, 0.25293537]) #0.21397742, 0.50635925, 0.67623288, 0.74856172, 
num_clusters_range = range(data.shape[0])  # Example range of number of clusters

result, result_new_labels = detect_label_switching(data, num_clusters_range)


previous iteration None
previous iteration [2 3 5 4 0 1]
num_cluster is 1
switched indices [2]
before changing previous iteration [2 3 5 4 0 1]
subiteration is 1
TEST (array([2], dtype=int64),)
[2]
give number of repeated label 1
previous iteration [2 3 1 4 0 1]
num_cluster is 2
switched indices [0 1 3 4]
before changing previous iteration [2 3 5 4 0 1]
subiteration is 0
TEST (array([0], dtype=int64),)
[0]
give number of repeated label 1
subiteration is 2
TEST (array([1], dtype=int64),)
[1]
give number of repeated label 1
subiteration is 3
TEST (array([3], dtype=int64),)
[3]
give number of repeated label 1
subiteration is 2
TEST (array([4], dtype=int64),)
[4]
give number of repeated label 1
previous iteration [0 2 1 3 2 1]
num_cluster is 3
switched indices [0 3 4]
before changing previous iteration [2 3 5 4 0 1]
subiteration is 2
TEST (array([0], dtype=int64),)
[0]
give number of repeated label 1
subiteration is 0
TEST (array([3], dtype=int64),)
[3]
give number of repeated label 1
subi

In [43]:
print(data)
print(result)
print(result_new_labels)

[0.59830198 0.7101328  0.25210179 0.90204906 0.79986231 0.25293537]
[[2 3 5 4 0 1]
 [2 3 1 4 0 1]
 [0 2 1 3 2 1]
 [2 2 1 0 0 1]
 [0 0 1 0 0 1]
 [0 0 0 0 0 0]]
[[2 3 5 4 0 1]
 [2 3 5 4 0 1]
 [2 3 5 4 0 1]
 [2 3 5 4 0 1]
 [2 3 5 4 0 1]
 [2 3 5 4 0 1]]


In [44]:
import numpy as np
import plotly.graph_objects as go
from sklearn.cluster import KMeans

def visualize_label_switching(data, num_clusters_range):
    prev_labels = None  # Initialize previous labels as None
    links = []

    for num_clusters in num_clusters_range:
        kmeans = KMeans(n_clusters=num_clusters)
        kmeans.fit(data.reshape(-1, 1))
        labels = kmeans.labels_

        if prev_labels is not None:
            # Compare current labels with previous labels
            switched_indices = np.where(labels != prev_labels)[0]
            if len(switched_indices) > 0:
                for index in switched_indices:
                    prev_label = prev_labels[index]
                    curr_label = labels[index]
                    links.append({"source": prev_label, "target": curr_label, "value": 1})

        prev_labels = labels

    # Create nodes
    nodes = [{"label": label} for label in np.unique(prev_labels)]

    # Create Sankey diagram
    fig = go.Figure(data=[go.Sankey(node=dict(label=node["label"]), link=dict(source=links["source"], target=links["target"], value=links["value"])) for node in nodes], layout_title_text="Label Switching Sankey Diagram")

    # Show the Sankey diagram
    fig.show()

# Example usage:
data = np.array([2, 4, 6, 10, 12, 15, 18, 20, 25, 30])
num_clusters_range = range(2, 6)  # Example range of number of clusters

visualize_label_switching(data, num_clusters_range)


c:\users\arthu\documents\swittz-epfl\phd\code\lib\site-packages\sklearn\cluster\_kmeans.py:870: FutureWarning:

The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning

c:\users\arthu\documents\swittz-epfl\phd\code\lib\site-packages\sklearn\cluster\_kmeans.py:870: FutureWarning:

The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning

c:\users\arthu\documents\swittz-epfl\phd\code\lib\site-packages\sklearn\cluster\_kmeans.py:870: FutureWarning:

The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning

c:\users\arthu\documents\swittz-epfl\phd\code\lib\site-packages\sklearn\cluster\_kmeans.py:870: FutureWarning:

The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning



TypeError: list indices must be integers or slices, not str

# Deterministic Sankey from method

In [65]:
# Generate sbm and adjacency matrix
import random
random.seed(420)
import sys
sys.path.append('..')
from pygraphon.graphons import Graphon
from pygraphon.graphons import StepGraphon
from experiments.utils_generators import AssociativeFullyRandom
k = 2
n = 40
# theta init
theta_intra_min, theta_intra_max = 0.5, 0.7
theta_inter_min, theta_inter_max = 0.1, 0.4
theta = AssociativeFullyRandom(
                    K=k,
                    min_inside_classes=theta_intra_min,
                    max_inside_classes=theta_intra_max,
                    min_outside_classes=theta_inter_min,
                    max_outside_classes=theta_inter_max,
                )
graphon = StepGraphon(theta.Theta, bandwidthHist=1 / k)
adjacency_matrix=graphon.draw(n=n, exchangeable=False)

In [66]:
#import smoothnethist
from pygraphon.estimators import SmoothNetHist

est = SmoothNetHist(bandwidth=1/(2*k))
est.fit(adjacency_matrix)
print(graphon.get_theta())

[[0.56667572 0.33863574]
 [0.33863574 0.51587328]]


In [67]:
list_graphons = [est.all_cluster_graphons[i].graphon[np.triu_indices(est.all_cluster_graphons[i].graphon.shape[0])] for i in range(len(est.all_cluster_graphons))]

sort_indices = np.argsort(list_graphons[-1])

for i in range(len(list_graphons)):
    list_graphons[i] = list_graphons[i][sort_indices]
graphons_array = np.array(list_graphons)
graphons_array = np.rot90(np.insert(graphons_array, 0, np.ones(graphons_array.shape[1]), axis=0), k=3)

In [68]:
list_graphons

[array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]),
 array([0.82710456, 0.82710456, 0.82710456, 0.82710456, 0.82710456,
        0.82710456, 0.82710456, 1.74921358, 1.74921358, 1.74921358]),
 array([0.62977859, 0.62977859, 0.62977859, 0.9890273 , 0.9890273 ,
        0.9890273 , 0.9890273 , 1.76604579, 1.76604579, 1.76604579]),
 array([0.44984185, 0.71974696, 0.71974696, 0.9890273 , 0.9890273 ,
        0.9890273 , 0.9890273 , 1.76604579, 1.76604579, 1.76604579]),
 array([0.45236786, 0.72378858, 0.72378858, 0.90473573, 0.90473573,
        1.08442629, 1.08442629, 1.77596272, 1.77596272, 1.77596272]),
 array([0.45236786, 0.67855179, 0.76902537, 0.90473573, 0.90473573,
        1.08442629, 1.08442629, 1.77596272, 1.77596272, 1.77596272]),
 array([0.45236786, 0.67855179, 0.76902537, 0.90473573, 0.90473573,
        1.08442629, 1.08442629, 1.75920836, 1.75920836, 1.80947145]),
 array([0.45297263, 0.67945895, 0.77005348, 0.90594527, 0.90594527,
        1.06448569, 1.10726644, 1.76156024, 1.76156024

In [69]:
graphons_array

array([[0.45297263, 0.45236786, 0.45236786, 0.45236786, 0.44984185,
        0.62977859, 0.82710456, 1.        , 1.        ],
       [0.67945895, 0.67855179, 0.67855179, 0.72378858, 0.71974696,
        0.62977859, 0.82710456, 1.        , 1.        ],
       [0.77005348, 0.76902537, 0.76902537, 0.72378858, 0.71974696,
        0.62977859, 0.82710456, 1.        , 1.        ],
       [0.90594527, 0.90473573, 0.90473573, 0.90473573, 0.9890273 ,
        0.9890273 , 0.82710456, 1.        , 1.        ],
       [0.90594527, 0.90473573, 0.90473573, 0.90473573, 0.9890273 ,
        0.9890273 , 0.82710456, 1.        , 1.        ],
       [1.06448569, 1.08442629, 1.08442629, 1.08442629, 0.9890273 ,
        0.9890273 , 0.82710456, 1.        , 1.        ],
       [1.10726644, 1.08442629, 1.08442629, 1.08442629, 0.9890273 ,
        0.9890273 , 0.82710456, 1.        , 1.        ],
       [1.76156024, 1.75920836, 1.77596272, 1.77596272, 1.76604579,
        1.76604579, 1.74921358, 1.        , 1.        ],


In [70]:
#makes list into pandas dataframe
import pandas as pd
pd.set_option('display.float_format', '{:.3f}'.format)
columns = []
for i in range(len(list_graphons)):
    columns.append("SmoothNetHist" + str(i))
columns.append("Count")

def list_to_df(array, columns):
    df = pd.DataFrame(array, columns=columns)
    return df

df = list_to_df(graphons_array, columns=columns)
df

,SmoothNetHist0,SmoothNetHist1,SmoothNetHist2,SmoothNetHist3,SmoothNetHist4,SmoothNetHist5,SmoothNetHist6,SmoothNetHist7,Count
0,0.453,0.452,0.452,0.452,0.450,0.630,0.827,1.000,1.000
1,0.679,0.679,0.679,0.724,0.720,0.630,0.827,1.000,1.000
2,0.770,0.769,0.769,0.724,0.720,0.630,0.827,1.000,1.000
3,0.906,0.905,0.905,0.905,0.989,0.989,0.827,1.000,1.000
4,0.906,0.905,0.905,0.905,0.989,0.989,0.827,1.000,1.000
5,1.064,1.084,1.084,1.084,0.989,0.989,0.827,1.000,1.000
6,1.107,1.084,1.084,1.084,0.989,0.989,0.827,1.000,1.000
7,1.762,1.759,1.776,1.776,1.766,1.766,1.749,1.000,1.000
8,1.762,1.759,1.776,1.776,1.766,1.766,1.749,1.000,1.000
9,1.812,1.809,1.776,1.776,1.766,1.766,1.749,1.000,1.000


# Plotting sankey

In [71]:
import sys
sys.path.insert(0, r"c:\users\arthu\documents\swittz-epfl\phd\code\lib\site-packages")
import plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [72]:
def find_peak_index(lst):
    for i in range(1, len(lst)):
        if lst[i] < lst[i - 1]:
            return i - 1
    return len(lst) - 1

# Example list
my_list = [1, 3, 5, 4, 2, 0]

peak_index = find_peak_index(my_list)
print("Index where the list stops increasing:", peak_index)


Index where the list stops increasing: 2


In [73]:
source_nodes = [0.259,0.583, 0.670, 0.912, 1.248, 1.253,1.685, 1.728, 1.872, 1.251, 1.705, 0.640, 1.754, 0.697, 0.605, 1.513, 1]
source_nodes = [x*est.bandwidth for x in source_nodes]
idx = find_peak_index(source_nodes)
# the 0.0001 is to avoid the error of having nodes one the boundaries of [0,1], somehow it messes up the visualization
source_intervals = [0.0001 if i < idx+1 else (i-idx)/len(source_nodes[idx+1:]) for i in range(len(source_nodes))]
nodes = [{"label": str(label)} for i, label in enumerate(sorted(source_nodes))]
intervals_x = [interval for _, interval in sorted(zip(source_nodes, source_intervals))]
intervals_x[intervals_x.index(1)] -= 0.0001
intervals_y = [i/len(nodes) for i in range(len(nodes))]
intervals_y[intervals_y.index(0)] += 0.0001
print(nodes)
print(intervals_y)
links = [
    {'source': 8, 'target': 9, 'value': 1},
    {'source': 10, 'target': 9, 'value': 1},
    {'source': 12, 'target': 13, 'value': 1},
    {'source': 14, 'target': 13, 'value': 1},
    {'source': 1, 'target': 3, 'value': 1},
    {'source': 4, 'target': 3, 'value': 1},
    {'source': 16, 'target': 15, 'value': 1},
    {'source': 13, 'target': 15, 'value': 2},
    {'source': 3, 'target': 5, 'value': 2},
    {'source': 6, 'target': 5, 'value': 1},
    {'source': 5, 'target': 2, 'value': 3},
    {'source': 0, 'target': 2, 'value': 1},
    {'source': 15, 'target': 11, 'value': 3},
    {'source': 9, 'target': 11, 'value': 2},
    {'source': 2, 'target': 7, 'value': 4},
    {'source': 11, 'target': 7, 'value': 5},
]

[{'label': '0.06475'}, {'label': '0.14575'}, {'label': '0.15125'}, {'label': '0.16'}, {'label': '0.1675'}, {'label': '0.17425'}, {'label': '0.228'}, {'label': '0.25'}, {'label': '0.312'}, {'label': '0.31275'}, {'label': '0.31325'}, {'label': '0.37825'}, {'label': '0.42125'}, {'label': '0.42625'}, {'label': '0.432'}, {'label': '0.4385'}, {'label': '0.468'}]
[0.0001, 0.058823529411764705, 0.11764705882352941, 0.17647058823529413, 0.23529411764705882, 0.29411764705882354, 0.35294117647058826, 0.4117647058823529, 0.47058823529411764, 0.5294117647058824, 0.5882352941176471, 0.6470588235294118, 0.7058823529411765, 0.7647058823529411, 0.8235294117647058, 0.8823529411764706, 0.9411764705882353]


In [74]:
import plotly.graph_objects as go
# Define the data

#sorted_links = sorted(links, key=lambda k: k['source']) #this doesnt chnage anything
# Create the figure and subplot
fig = make_subplots(rows=1, cols=1)

# Add the Sankey trace
fig.add_trace(go.Sankey(
    arrangement = "perpendicular",
    node=dict(
        pad=10*len(nodes),  # Adjust the padding between nodes
        thickness=20,  # Adjust the thickness of nodes
        line=dict(color='black', width=1),  # Define the color and width of node lines
        label=[node['label'] for node in nodes],  # Extract node labels from the 'nodes' list
        x = intervals_x,
        y = intervals_y,
    ),
    link=dict(
        source=[link['source'] for link in links],  # Extract source indices from the 'links' list
        target=[link['target'] for link in links],  # Extract target indices from the 'links' list
        value=[link['value'] for link in links],  # Extract values from the 'links' list
    )
))

# Update the layout
fig.update_layout(title_text='Sankey Diagram')

# Show the plot
fig.show()

